In [1]:
RUN_TARGET = "colab"  # "colab" | "local"


## Setup Instructions

Notebook 06 generates residue-level probe rows for supported active model families. It is registry-based rather than fully model-agnostic: unknown checkpoints are skipped, and retired top-1-unfrozen baseline checkpoints are intentionally excluded. Protein-level classification metrics are not generated here; notebook 07 loads them later from saved JSON artifacts.

### Typical workflow
1. Train or update a supported checkpoint in notebook 03, 04, or 05.
2. Run this notebook to generate or refresh saved probe-row CSVs in `results/probing/rows/`.
3. Open `07_compare_all_model_probes.ipynb` to build paper figures and tables from those saved artifacts.


In [2]:
if RUN_TARGET == "colab":
    import importlib.metadata as _md
    import subprocess as _sp
    import sys as _sys

    _required = {
        "numpy": "1.26.4",
        "pandas": "2.3.2",
        "scikit-learn": "1.8.0",
        "transformers": "4.48.1",
        "huggingface-hub": "0.36.2",
        "seaborn": "0.13.2",
    }

    def _version_matches(name: str, expected: str) -> bool:
        try:
            return _md.version(name) == expected
        except _md.PackageNotFoundError:
            return False

    _missing_or_mismatched = [
        f"{name}=={version}"
        for name, version in _required.items()
        if not _version_matches(name, version)
    ]

    if _missing_or_mismatched:
        print("Installing:", ", ".join(_missing_or_mismatched))
        _sp.run([
            _sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--upgrade",
            *_missing_or_mismatched,
        ], check=True)
        raise SystemExit(
            "Colab environment updated. Restart the runtime once, then rerun the notebook from the top."
        )
    else:
        print("Colab environment already compatible. No reinstall needed.")
else:
    print("Local environment detected. Skipping Colab setup.")


Installing: numpy==1.26.4, pandas==2.3.2, scikit-learn==1.8.0, transformers==4.48.1, huggingface-hub==0.36.2


KeyboardInterrupt: 

In [ ]:
if RUN_TARGET == "colab":
    from google.colab import drive as _drive
    from pathlib import Path

    _drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/XAllergen2.0")
    DRIVE_MODELS = DRIVE_ROOT / "models"
    DRIVE_RESULTS = DRIVE_ROOT / "results"
    print(f"Google Drive mounted: {DRIVE_ROOT}")
else:
    print("Local run: skipping Google Drive mount.")


# 06 - Registry-Driven Probe Row Generation

This notebook centralizes residue-level probe generation for the supported active model registry. It is semi-automatic over known model families, not a generic sweep over arbitrary checkpoint names. Unknown checkpoints are skipped, retired top-1-unfrozen baselines are ignored, and supported MTL probe rows are generated against the frozen baseline reference.


In [ ]:
import sys
from pathlib import Path

for _candidate in [Path.cwd(), *Path.cwd().parents]:
    _src_dir = _candidate / "src"
    if (_src_dir / "xallergen").exists() and str(_src_dir) not in sys.path:
        sys.path.insert(0, str(_src_dir))
        break
import importlib
import json

if RUN_TARGET == "colab":
    RUNTIME_ROOT = Path("/content/XAllergen2.0")
    if str(RUNTIME_ROOT) not in sys.path:
        sys.path.insert(0, str(RUNTIME_ROOT))
    _SRC_DIR = RUNTIME_ROOT / "src"
    if _SRC_DIR.exists() and str(_SRC_DIR) not in sys.path:
        sys.path.insert(0, str(_SRC_DIR))

import xallergen.baseline_notebook_utils as baseline_notebook_utils
import xallergen.deep_plant_allergy_utils as deep_plant_allergy_utils
import xallergen.mtl_epitope_notebook_utils as mtl_epitope_notebook_utils
import xallergen.plotting_paper_figures as plotting_paper_figures

importlib.reload(baseline_notebook_utils)
importlib.reload(deep_plant_allergy_utils)
importlib.reload(mtl_epitope_notebook_utils)
importlib.reload(plotting_paper_figures)

from xallergen.baseline_notebook_utils import (
    DROPOUT,
    HF_MODEL_NAME,
    HIDDEN_DIM,
    IG_STEPS,
    build_tokenizer,
    configure_matplotlib_cache,
    detect_device,
    find_project_root,
    load_baseline_checkpoint,
    load_mtl_checkpoint,
    prepare_baseline_probe_frame,
    print_runtime_context,
    run_baseline_probe_suite,
)
from xallergen.deep_plant_allergy_utils import (
    build_embedding_model,
    build_tokenizer as build_deep_plant_tokenizer,
    load_deep_plant_allergy_checkpoint,
    run_deep_plant_probe_suite,
)
from xallergen.mtl_epitope_notebook_utils import (
    get_retired_checkpoint_names,
    get_supported_probe_model_registry,
    run_probe_suite,
    summarize_probe_outputs,
)
from xallergen.plotting_paper_figures import (
    build_output_paths_for_supported_mtl,
    save_registry_probe_summary,
)

if RUN_TARGET != "colab":
    configure_matplotlib_cache(Path.cwd())

import pandas as pd
import torch


In [ ]:
if RUN_TARGET == "colab":
    PROJECT_ROOT = RUNTIME_ROOT
    MODELS_DIR = DRIVE_MODELS if DRIVE_MODELS.exists() else PROJECT_ROOT / "models"
    RESULTS_DIR = DRIVE_RESULTS if DRIVE_RESULTS.exists() else PROJECT_ROOT / "results"
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using models dir: {MODELS_DIR}")
    print(f"Using results dir: {RESULTS_DIR}")
    print(f"Device: {DEVICE}")
else:
    PROJECT_ROOT = find_project_root(Path.cwd())
    DEVICE = detect_device()
    print_runtime_context(DEVICE, PROJECT_ROOT)
    MODELS_DIR = PROJECT_ROOT / "models"
    RESULTS_DIR = PROJECT_ROOT / "results"

DATA_DIR = PROJECT_ROOT / "data"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
for _output_subdir in [
    RESULTS_DIR / "classification",
    RESULTS_DIR / "probing" / "rows",
    RESULTS_DIR / "probing" / "summaries",
    RESULTS_DIR / "figures" / "diagnostics",
]:
    _output_subdir.mkdir(parents=True, exist_ok=True)

POSITIVES_CSV = DATA_DIR / "positives_splitB.csv"
tokenizer = build_tokenizer(HF_MODEL_NAME)
epitope_probe_df = prepare_baseline_probe_frame(POSITIVES_CSV)
print(f"Probe proteins in splitB: {len(epitope_probe_df)}")


## Generation Parameters

- `OVERWRITE_EXISTING` keeps existing probe artifacts when set to `False`.
- `N_RANDOM_DRAWS` controls the random-null estimate written into the probe rows.
- `IG_INTERNAL_BATCH_SIZE` controls the memory/speed tradeoff for Captum IG and SmoothGrad-IG.


In [ ]:
OVERWRITE_EXISTING = True
N_RANDOM_DRAWS = 100
IG_INTERNAL_BATCH_SIZE = 1
SMOOTHGRAD_IG_SAMPLES = 10
SMOOTHGRAD_IG_NOISE_STD = 0.05


## Supported Registry

The supported-model registry defines which active model families notebook 06 knows how to probe. Unknown checkpoints are skipped rather than inferred automatically. Optional `MTL ESM-2 top-1` is supplementary only, and retired top-1-unfrozen baseline checkpoints are intentionally excluded.


In [ ]:
retired_checkpoint_names = set(get_retired_checkpoint_names())
supported_registry = get_supported_probe_model_registry(
    models_dir=MODELS_DIR,
    results_dir=RESULTS_DIR,
    include_supplementary=True,
)
supported_lookup = {spec["checkpoint_name"]: spec for spec in supported_registry}
registry_df = pd.DataFrame(
    [
        {
            "family_key": spec["family_key"],
            "display_label": spec["display_label"],
            "checkpoint_name": spec["checkpoint_name"],
            "checkpoint_exists": spec["checkpoint_exists"],
            "probe_rows_path": str(spec["probe_rows_path"]),
            "summary_path": str(spec["summary_path"]),
            "supplementary_only": spec["supplementary_only"],
            "supported_methods": ", ".join(spec["supported_methods"]),
        }
        for spec in supported_registry
    ]
)
registry_df


## Probe Generation

The frozen baseline probe rows are generated first and then reused by supported MTL families. DeepPlantAllergy and supported MTL checkpoints are generated only if they match the registry. Retired baseline checkpoints are skipped explicitly, and unsupported checkpoint names are recorded as unknown.


In [ ]:
generation_records = []
deep_plant_embedding_model = None
deep_plant_tokenizer = None
baseline_probe_df = None

def _available_methods_from_rows(probe_rows_path: Path) -> str:
    if not Path(probe_rows_path).exists():
        return ""
    probe_df = pd.read_csv(probe_rows_path, usecols=["method"])
    methods = sorted(str(value) for value in probe_df["method"].dropna().unique())
    return ", ".join(methods)

def _append_record(spec: dict, status: str, probe_rows_path: Path | None = None) -> None:
    generation_records.append(
        {
            "checkpoint_name": spec["checkpoint_name"],
            "family_key": spec["family_key"],
            "display_label": spec["display_label"],
            "status": status,
            "available_methods": _available_methods_from_rows(probe_rows_path or spec["probe_rows_path"]),
            "probe_rows_path": str(probe_rows_path or spec["probe_rows_path"]),
        }
    )

baseline_spec = next(spec for spec in supported_registry if spec["family_key"] == "baseline")
if baseline_spec["probe_rows_exists"] and not OVERWRITE_EXISTING:
    baseline_probe_df = pd.read_csv(baseline_spec["probe_rows_path"])
    baseline_probe_df["model_family"] = baseline_spec["display_label"]
    print(f"Reusing baseline probe rows: {baseline_spec['probe_rows_path']}")
    _append_record(baseline_spec, "reused")
elif baseline_spec["checkpoint_exists"]:
    baseline_model, _ = load_baseline_checkpoint(baseline_spec["checkpoint_path"], DEVICE)
    baseline_probe_df = run_baseline_probe_suite(
        model=baseline_model,
        tokenizer=tokenizer,
        eval_df=epitope_probe_df,
        device=DEVICE,
        ig_steps=IG_STEPS,
        n_random_draws=N_RANDOM_DRAWS,
        ig_internal_batch_size=IG_INTERNAL_BATCH_SIZE,
        smoothgrad_ig_samples=SMOOTHGRAD_IG_SAMPLES,
        smoothgrad_ig_noise_std=SMOOTHGRAD_IG_NOISE_STD,
        include_shuffled_mean=False,
    )
    baseline_probe_df["model_family"] = baseline_spec["display_label"]
    baseline_probe_df.to_csv(baseline_spec["probe_rows_path"], index=False)
    save_registry_probe_summary(
        baseline_probe_df,
        baseline_spec["summary_path"],
        baseline_spec["supported_methods"],
    )
    print(f"Saved baseline probe rows to: {baseline_spec['probe_rows_path']}")
    _append_record(baseline_spec, "generated")
else:
    print(f"Skipping missing supported checkpoint: {baseline_spec['checkpoint_name']}")
    _append_record(baseline_spec, "skipped_missing")

for spec in supported_registry:
    if spec["family_key"] == "baseline":
        continue
    if spec["probe_rows_exists"] and not OVERWRITE_EXISTING:
        print(f"Reusing probe rows for {spec['display_label']}: {spec['probe_rows_path']}")
        _append_record(spec, "reused")
        continue
    if not spec["checkpoint_exists"]:
        print(f"Skipping missing supported checkpoint: {spec['checkpoint_name']}")
        _append_record(spec, "skipped_missing")
        continue
    if spec["model_kind"] == "deep_plant":
        deep_plant_model, deep_plant_metadata = load_deep_plant_allergy_checkpoint(spec["checkpoint_path"], DEVICE)
        deep_plant_hf_model_name = deep_plant_metadata.get("hf_model_name")
        if deep_plant_embedding_model is None:
            deep_plant_tokenizer = build_deep_plant_tokenizer(deep_plant_hf_model_name)
            deep_plant_embedding_model = build_embedding_model(deep_plant_hf_model_name, device=DEVICE)
        probe_df = run_deep_plant_probe_suite(
            model=deep_plant_model,
            embedding_model=deep_plant_embedding_model,
            tokenizer=deep_plant_tokenizer,
            eval_df=epitope_probe_df,
            device=DEVICE,
            ig_steps=IG_STEPS,
            n_random_draws=N_RANDOM_DRAWS,
        )
        probe_df["model_family"] = spec["display_label"]
        probe_df.to_csv(spec["probe_rows_path"], index=False)
        save_registry_probe_summary(
            probe_df,
            spec["summary_path"],
            spec["supported_methods"],
        )
        _append_record(spec, "generated")
        continue
    output_paths = build_output_paths_for_supported_mtl(
        family_key=spec["family_key"],
        display_label=spec["display_label"],
        models_dir=MODELS_DIR,
        results_dir=RESULTS_DIR,
        baseline_checkpoint_path=baseline_spec["checkpoint_path"],
        baseline_summary_path=baseline_spec["summary_path"],
    )
    model, _ = load_mtl_checkpoint(
        spec["checkpoint_path"],
        DEVICE,
        model_name=HF_MODEL_NAME,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
    )
    probe_outputs = run_probe_suite(
        model=model,
        tokenizer=tokenizer,
        epitope_probe_df=epitope_probe_df,
        baseline_checkpoint_path=baseline_spec["checkpoint_path"],
        device=DEVICE,
        hidden_dim=HIDDEN_DIM,
        dropout=DROPOUT,
        output_paths=output_paths,
        ig_steps=IG_STEPS,
        n_random_draws=N_RANDOM_DRAWS,
        ig_internal_batch_size=IG_INTERNAL_BATCH_SIZE,
        resume=False,
        precomputed_baseline_probe_df=baseline_probe_df,
    )
    summarize_probe_outputs(
        probe_df=probe_outputs["probe_df"],
        baseline_probe_df=probe_outputs["baseline_probe_df"],
        output_paths=output_paths,
    )
    _append_record(spec, "generated", probe_rows_path=output_paths.probe_rows_path)

processed_checkpoint_names = {spec["checkpoint_name"] for spec in supported_registry}
for checkpoint_path in sorted(MODELS_DIR.glob("*.pt")):
    if checkpoint_path.name in processed_checkpoint_names:
        continue
    if checkpoint_path.name in retired_checkpoint_names:
        print(f"Skipping retired baseline checkpoint: {checkpoint_path.name}")
        generation_records.append(
            {
                "checkpoint_name": checkpoint_path.name,
                "family_key": "retired",
                "display_label": "Retired baseline",
                "status": "skipped_retired",
                "available_methods": "",
                "probe_rows_path": "",
            }
        )
        continue
    print(f"Skipping unknown checkpoint: {checkpoint_path.name}")
    generation_records.append(
        {
            "checkpoint_name": checkpoint_path.name,
            "family_key": "unknown",
            "display_label": "Unsupported checkpoint",
            "status": "skipped_unknown",
            "available_methods": "",
            "probe_rows_path": "",
        }
    )

generation_df = pd.DataFrame(generation_records)
generation_df


## Generation Summary

The final summary table reports the checkpoint name, registry family key, publication label, generation status, methods present in the saved row artifact, and the probe-row output path.


In [ ]:
summary_columns = [
    "checkpoint_name",
    "family_key",
    "display_label",
    "status",
    "available_methods",
    "probe_rows_path",
]
print(generation_df[summary_columns].to_string(index=False))
